Reloaded MNIST model

In [120]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
%matplotlib inline

# Model Class
class ConvolutionalNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1,6,3,1)
    self.conv2 = nn.Conv2d(6,16,3,1)
    # Fully Connected Layer
    self.fc1 = nn.Linear(5*5*16, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10)

  def forward(self, X):
    X = F.relu(self.conv1(X))
    X = F.max_pool2d(X,2,2) # 2x2 kernal and stride 2
    # Second Pass
    X = F.relu(self.conv2(X))
    X = F.max_pool2d(X,2,2) # 2x2 kernal and stride 2

    # Re-View to flatten it out
    X = X.view(-1, 16*5*5) # negative one so that we can vary the batch size

    # Fully Connected Layers
    X = F.relu(self.fc1(X))
    X = F.relu(self.fc2(X))
    X = self.fc3(X)
    return F.log_softmax(X, dim=1)

# Initialize a new model instance with the same architecture as the saved model
new_model = ConvolutionalNetwork()

# Load the saved model's state dictionary into the new model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
new_model.load_state_dict(torch.load('mnist_model_weights.pth'))

# Set the model to evaluation mode
new_model.eval()


ConvolutionalNetwork(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [121]:
# Import my Paint files and test...
from PIL import Image

# Load the image you saved from Paint
img = Image.open('five_28.png')

# Convert the image to grayscale ('L' mode)
img = img.convert('L')

In [122]:
img

In [123]:
print(type(img))

<class 'PIL.Image.Image'>


In [124]:
# Define a transform to convert the PIL image to a tensor and normalize it
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts to Tensor and scales pixels between 0 and 1 (0 to 255 becomes 0.0 to 1.0)
    # transforms.Lambda(lambda x: 1 - x),  # Inverts the pixel value
    # transforms.Normalize((0.5,), (0.5,))  # Normalize to have mean 0.5 and standard deviation 0.5
])

# Apply the transform to the image
img_tensor = transform(img)

# Check the size of the tensor to ensure it's correct
print(img_tensor.shape)  # Should print torch.Size([1, 28, 28])

# Print the tensor's type
print(type(img_tensor))
img_tensor.shape

torch.Size([1, 28, 28])
<class 'torch.Tensor'>


torch.Size([1, 28, 28])

In [125]:
img_tensor

tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0078, 0.0039, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0039, 0.0000, 0.0000,

In [126]:
# Pass the image thru our model
new_model.eval()
with torch.no_grad():
  new_prediction = new_model(img_tensor.view(1,1,28,28)) # batch size of 1, 1 color channel, 28x28 image

In [127]:
print(new_prediction)

tensor([[-2.7552e+01, -2.6861e+01, -3.4278e+01, -1.4815e+01, -2.5973e+01,
         -5.9605e-07, -1.8501e+01, -3.4312e+01, -1.7638e+01, -1.5402e+01]])


In [128]:
new_prediction.argmax()

tensor(5)